In [0]:
from pyspark.sql.functions import col, from_json, when
from pyspark.sql.types import StructType, StringType, DoubleType

df_txn = spark.table("workspace.gold_sales_bronze.raw_transactions")
df_branch = spark.table("workspace.gold_sales_bronze.branch_master")

audit_schema = StructType() \
    .add("lab_name", StringType()) \
    .add("purity", DoubleType()) \
    .add("inspector", StringType())


df_silver = df_txn \
    .join(df_branch, on="branch_id", how="left") \
    .withColumn("audit_parsed", from_json(col("audit_metadata_json"), audit_schema)) \
    .select("*", "audit_parsed.*") \
    .drop("audit_parsed") \
    .withColumn(
        "gross_revenue_usd",
        col("weight_troy_oz") * col("spot_price_usd_oz") * (1 + col("premium_pct") / 100)
    ) \
    .withColumn(
        "gold_category",
        when(col("product_type").like("%bar%"), "Bar")
        .when(col("product_type").like("%coin%"), "Coin")
        .when(col("product_type").like("%jewellery%"), "Jewellery")
        .otherwise("Other")
    )

In [0]:
from delta.tables import DeltaTable

target_table = "workspace.gold_sales_silver.enriched_transactions"
if spark.catalog.tableExists(target_table):

    delta_table = DeltaTable.forName(spark, target_table)

    delta_table.alias("t").merge(
        df_silver.alias("s"),
        "t.transaction_id = s.transaction_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
else:
    df_silver.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)